# 1. Build the lake time series from raw satellite data

Turns the raw Sentinel-1 / Sentinel-2 NetCDF into the per-lake daily time series
that every experiment consumes.

**This notebook is optional.** The CSVs it produces already ship in
`data/processed/`. Run it to regenerate them from the raw archive, or to adapt
the pipeline.

**Prerequisite:** `python data/download_data.py`

| Input | |
| --- | --- |
| `data/raw/all_lakes_2019.nc` | daily `HV_lake`, `HV_out`, `S2_water` for 6,146 lakes |
| `data/raw/all_training.geojson` | 1,000 manually labeled lake outlines |

| Output | |
| --- | --- |
| $HV_{anom}$ | 365 daily values per lake, 1,000 lakes |
| $p_{water}$ | 365 daily values per lake, 1,000 lakes |

The repository ships the 777-lake evaluation and training splits derived from
these; the 1,000-lake tables are an intermediate product.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "src" / "rpsgmm").is_dir():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError("Run this notebook from inside the repository.")
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

print("Repository root:", REPO_ROOT)

In [ ]:
import numpy as np
import pandas as pd

from build_processed_data import build_backscatter, build_water, load_raw

RAW = REPO_ROOT / "data" / "raw"
PROCESSED = REPO_ROOT / "data" / "processed"

NETCDF = RAW / "all_lakes_2019.nc"
GEOJSON = RAW / "all_training.geojson"

if not NETCDF.exists():
    raise FileNotFoundError(
        f"{NETCDF} is missing. Fetch it first:\n"
        "    python data/download_data.py"
    )

## Load the labeled lakes

`load_raw` keeps only the lakes that carry a manual label, and sorts them by ID
so the row order is deterministic.

In [ ]:
ids, hv_lake, hv_out, s2_water, labels = load_raw(NETCDF, GEOJSON)

print(f"{len(ids)} labeled lakes x {hv_lake.shape[1]} days")
print(f"first few IDs: {list(ids[:4])}")
print()
print(pd.Series(list(labels.values())).value_counts().to_string())

## Backscatter anomaly

Sentinel-1 revisits a given lake from several orbits, and the returned signal
differs systematically between them. Section 4.1 of the paper applies a **12-day
smoothing filter** to suppress that orbit-to-orbit variability, then linearly
interpolates across revisit gaps, and finally differences the in-lake signal
against its surrounding buffer (Equation 9):

$$HV_{anom} = HV_{lake} - HV_{background}$$

The order matters: smoothing runs **before** interpolation, so interpolated
values do not feed back into the filter.

> **Order matters.** Smoothing is applied before interpolation, so interpolated
> values never feed back into the filter. The verification cell at the end of
> this notebook confirms the result matches the shipped CSVs.

In [ ]:
backscatter = build_backscatter(hv_lake, hv_out)

print("backscatter:", backscatter.shape)
print(f"range: [{backscatter.min():.3f}, {backscatter.max():.3f}] dB")
print(f"remaining NaNs: {int(np.isnan(backscatter).sum())}")

## Water percentage

For Sentinel-2 the fraction of in-lake pixels classified as water (Equation 10):

$$p_{water} = \frac{N_{water}}{N_{total}} \times 100\%$$

Gaps *between* observations are interpolated. Gaps before the first and after
the last observation are set to zero rather than being padded with the nearest
value: outside the observed window the lake is treated as having no detected
surface water.

In [ ]:
water = build_water(s2_water)

print("water:", water.shape)
print(f"range: [{water.min():.3f}, {water.max():.3f}]")
print(f"remaining NaNs: {int(np.isnan(water).sum())}")

## Assemble and save

In [ ]:
days = range(1, 366)
index = pd.Index(ids, name="ids")

backscatter_frame = pd.DataFrame(
    backscatter, index=index, columns=[f"backscatter_diff_{d}" for d in days]
)
water_frame = pd.DataFrame(
    water, index=index, columns=[f"water_percentage_{d}" for d in days]
)
combined = backscatter_frame.join(water_frame)

backscatter_frame.head()

In [ ]:
# To save these, use the command-line equivalent, which writes to an
# untracked directory:
#     python scripts/build_processed_data.py --write

print("backscatter:", backscatter_frame.shape)
print("combined   :", combined.shape)

## Verify against the shipped data

The released CSVs are the exact arrays behind the published results, so a
rebuild must match them. This is also available as
`python scripts/build_processed_data.py --check`.

In [ ]:
shipped = pd.read_csv(PROCESSED / "backscatter_777.csv").set_index("ids")
columns = [f"backscatter_diff_{d}" for d in range(121, 365)]
common = backscatter_frame.index.intersection(shipped.index)

difference = np.abs(
    backscatter_frame.loc[common, columns].to_numpy(float)
    - shipped.loc[common, columns].to_numpy(float)
)
matching = int((difference.max(axis=1) <= 1e-6).sum())

print(f"{matching} / {len(common)} evaluation lakes match")
print(f"max absolute difference: {difference.max():.2e}")

---

Next: [`02_lake_selection.ipynb`](02_lake_selection.ipynb) narrows these 1,000
lakes to the 777 used in the paper.